# Project V1 Correct - 2026_H1 Weekly Prediction Diagnostic

이 노트북은 `2026_H1` 기준으로 아래 흐름이 실제로 연결되는지 점검하기 위한 진단용 노트북입니다.

1. Streamlit이 어떤 파일/분기에서 예측 결과를 읽는지 확인
2. 주간 자동 수집 산출물이 존재하는지 확인
3. 피처 생성에 필요한 입력이 채워져 있는지 확인
4. 예측 파일이 생성됐는지, 그 안에 `strong_out`이 실제로 있는지 확인
5. 필요하면 로컬에서 수집/예측 스크립트를 실행할 명령까지 정리

이 노트북은 코드 수정이 아니라 “현재 상태 점검”을 우선 목적으로 합니다.


In [2]:
from pathlib import Path
import json
import ast
import re

import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()
AUTO_DIR = BASE_DIR / "data" / "incoming" / "auto"
RAW_SQL = BASE_DIR / "data" / "raw" / "kospi_db_full_20260320.sql"
TARGET_PERIOD = "2026_H1"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

print("BASE_DIR:", BASE_DIR)
print("AUTO_DIR exists:", AUTO_DIR.exists())
print("RAW_SQL exists:", RAW_SQL.exists())
print("TARGET_PERIOD:", TARGET_PERIOD)


BASE_DIR: c:\Users\Admin\Desktop\Project\Next200\Next200_v1
AUTO_DIR exists: True
RAW_SQL exists: True
TARGET_PERIOD: 2026_H1


## 1. Streamlit이 참조하는 2026_H1 예측 파일 확인

앱은 미래 반기 예측일 때 `data/incoming/auto/weekly_predictions_{period}.csv`를 우선 보는 구조입니다.
아래에서 실제 파일 존재 여부를 먼저 확인합니다.


In [3]:
expected_files = [
    AUTO_DIR / f"weekly_predictions_{TARGET_PERIOD}.csv",
    AUTO_DIR / f"weekly_strong_in_{TARGET_PERIOD}.csv",
    AUTO_DIR / f"weekly_strong_out_{TARGET_PERIOD}.csv",
    AUTO_DIR / "weekly_prediction_summary.json",
    AUTO_DIR / "weekly_collection_summary.json",
    AUTO_DIR / "yahoo_price_daily.csv",
    AUTO_DIR / "naver_stock_meta_weekly.csv",
    AUTO_DIR / "dart_major_holder_weekly.csv",
    AUTO_DIR / "naver_foreign_holding_weekly.csv",
]

status_rows = []
for path in expected_files:
    status_rows.append(
        {
            "file": str(path.relative_to(BASE_DIR)),
            "exists": path.exists(),
            "size_bytes": path.stat().st_size if path.exists() else np.nan,
            "modified": pd.Timestamp(path.stat().st_mtime, unit="s").tz_localize("UTC").tz_convert("Asia/Seoul")
            if path.exists()
            else pd.NaT,
        }
    )

file_status_df = pd.DataFrame(status_rows)
file_status_df


,file,exists,size_bytes,modified
0,data\incoming\auto\weekly_predictions_2026_H1.csv,False,NaN,NaT
1,data\incoming\auto\weekly_strong_in_2026_H1.csv,False,NaN,NaT
2,data\incoming\auto\weekly_strong_out_2026_H1.csv,False,NaN,NaT
3,data\incoming\auto\weekly_prediction_summary.json,False,NaN,NaT
4,data\incoming\auto\weekly_collection_summary.json,False,NaN,NaT
5,data\incoming\auto\yahoo_price_daily.csv,False,NaN,NaT
6,data\incoming\auto\naver_stock_meta_weekly.csv,False,NaN,NaT
7,data\incoming\auto\dart_major_holder_weekly.csv,False,NaN,NaT
8,data\incoming\auto\naver_foreign_holding_weekl...,False,NaN,NaT


## 2. 앱 코드 기준으로 `strong_out`이 비어 보일 수 있는 조건

앱 화면에서는 `strong_out.empty`이면 편출 패널에 경고만 띄웁니다.
즉, 아래 둘 중 하나면 편출 결과가 안 보입니다.

- `weekly_predictions_2026_H1.csv` 자체가 없음
- 파일은 있지만 `strong_out == 1`인 행이 없음


In [4]:
scored_path = AUTO_DIR / f"weekly_predictions_{TARGET_PERIOD}.csv"

if not scored_path.exists():
    print("현재 weekly_predictions 파일이 없습니다.")
else:
    scored_df = pd.read_csv(scored_path)
    scored_df.columns = [str(c) for c in scored_df.columns]
    for col in ["strong_in", "strong_out", "pred_top200", "pred_rank", "score"]:
        if col in scored_df.columns:
            scored_df[col] = pd.to_numeric(scored_df[col], errors="coerce")

    summary_frame = pd.DataFrame(
        {
            "metric": [
                "rows",
                "pred_top200 count",
                "strong_in count",
                "strong_out count",
                "min strong_out score",
                "max strong_out score",
            ],
            "value": [
                len(scored_df),
                int(scored_df["pred_top200"].fillna(0).eq(1).sum()) if "pred_top200" in scored_df.columns else np.nan,
                int(scored_df["strong_in"].fillna(0).eq(1).sum()) if "strong_in" in scored_df.columns else np.nan,
                int(scored_df["strong_out"].fillna(0).eq(1).sum()) if "strong_out" in scored_df.columns else np.nan,
                scored_df.loc[scored_df["strong_out"].fillna(0).eq(1), "score"].min() if {"strong_out", "score"}.issubset(scored_df.columns) else np.nan,
                scored_df.loc[scored_df["strong_out"].fillna(0).eq(1), "score"].max() if {"strong_out", "score"}.issubset(scored_df.columns) else np.nan,
            ],
        }
    )
    display(summary_frame)


현재 weekly_predictions 파일이 없습니다.


## 3. 2026_H1 기간 자체는 period 테이블에 정의돼 있는지 확인

`build_weekly_features()`는 `period` 테이블에서 활성 반기를 찾습니다.
따라서 `2026_H1` 정의가 없으면 주간 예측이 애초에 그 기간으로 가지 못합니다.


In [5]:
def load_sql_insert_table(sql_text: str, table_name: str, columns: list[str]) -> pd.DataFrame:
    rows = []
    pattern = rf"INSERT INTO `{table_name}` VALUES (.*?);\n"
    for match in re.finditer(pattern, sql_text, flags=re.S):
        raw = "[" + match.group(1).replace("NULL", "None") + "]"
        rows.extend(ast.literal_eval(raw))
    return pd.DataFrame(rows, columns=columns)


sql_text = RAW_SQL.read_text(encoding="utf-8", errors="ignore")
period_df = load_sql_insert_table(sql_text, "period", ["period", "period_start", "period_end"])
period_df["period_start"] = pd.to_datetime(period_df["period_start"])
period_df["period_end"] = pd.to_datetime(period_df["period_end"])
period_df.loc[period_df["period"].astype(str) == TARGET_PERIOD]


,period,period_start,period_end
12,2026_H1,2025-11-01,2026-04-30


## 4. 주간 자동 수집 입력 파일 상태 확인

`build_weekly_features()`는 아래 입력을 사용합니다.

- `yahoo_price_daily.csv`
- `naver_stock_meta_weekly.csv`
- `dart_major_holder_weekly.csv`
- `naver_foreign_holding_weekly.csv`

여기서 `price` 또는 `meta`가 비어 있으면 피처 생성이 실패합니다.


In [6]:
def safe_read_csv(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


input_frames = {
    "yahoo_price_daily": safe_read_csv(AUTO_DIR / "yahoo_price_daily.csv", dtype={"ticker": str}),
    "naver_stock_meta_weekly": safe_read_csv(AUTO_DIR / "naver_stock_meta_weekly.csv", dtype={"ticker": str}),
    "dart_major_holder_weekly": safe_read_csv(AUTO_DIR / "dart_major_holder_weekly.csv", dtype={"ticker": str}),
    "naver_foreign_holding_weekly": safe_read_csv(AUTO_DIR / "naver_foreign_holding_weekly.csv", dtype={"ticker": str}),
}

input_status_rows = []
for name, frame in input_frames.items():
    input_status_rows.append(
        {
            "name": name,
            "rows": len(frame),
            "cols": len(frame.columns),
            "has_ticker": "ticker" in frame.columns,
            "sample_cols": ", ".join(frame.columns[:8]) if len(frame.columns) else "",
        }
    )

input_status_df = pd.DataFrame(input_status_rows)
input_status_df


,name,rows,cols,has_ticker,sample_cols
0,yahoo_price_daily,0,0,False,
1,naver_stock_meta_weekly,0,0,False,
2,dart_major_holder_weekly,0,0,False,
3,naver_foreign_holding_weekly,0,0,False,


In [7]:
for name, frame in input_frames.items():
    print("\n###", name)
    if frame.empty:
        print("EMPTY")
        continue
    display(frame.head(3))



### yahoo_price_daily
EMPTY

### naver_stock_meta_weekly
EMPTY

### dart_major_holder_weekly
EMPTY

### naver_foreign_holding_weekly
EMPTY


## 5. 주간 예측 출력 파일이 있다면 `strong_in`, `strong_out` 상세 확인

여기서 핵심은:

- `strong_out` 컬럼이 실제로 존재하는지
- `strong_out == 1` 행이 있는지
- 그 종목들이 어떤 종목인지


In [8]:
if not scored_path.exists():
    print("weekly_predictions 파일이 없어서 strong_out 상세를 확인할 수 없습니다.")
else:
    scored_df = pd.read_csv(scored_path, dtype={"ticker": str})
    scored_df["ticker"] = scored_df["ticker"].astype(str).str.zfill(6)
    for col in ["score", "pred_rank", "period_rank", "pred_top200", "strong_in", "strong_out"]:
        if col in scored_df.columns:
            scored_df[col] = pd.to_numeric(scored_df[col], errors="coerce")

    strong_in_df = scored_df.loc[scored_df["strong_in"].fillna(0).eq(1)].copy() if "strong_in" in scored_df.columns else pd.DataFrame()
    strong_out_df = scored_df.loc[scored_df["strong_out"].fillna(0).eq(1)].copy() if "strong_out" in scored_df.columns else pd.DataFrame()

    print("strong_in rows:", len(strong_in_df))
    print("strong_out rows:", len(strong_out_df))
    display(strong_in_df[["ticker", "company", "score", "pred_rank"]].head(20) if not strong_in_df.empty else strong_in_df)
    display(strong_out_df[["ticker", "company", "score", "pred_rank"]].head(20) if not strong_out_df.empty else strong_out_df)


weekly_predictions 파일이 없어서 strong_out 상세를 확인할 수 없습니다.


## 6. 앱이 읽는 소스와 노트북 진단 결과를 연결해서 해석하기

아래 셀은 현재 상태를 사람이 읽기 좋은 메시지로 요약합니다.


In [9]:
messages = []

if not (AUTO_DIR / f"weekly_predictions_{TARGET_PERIOD}.csv").exists():
    messages.append("1. 2026_H1 weekly_predictions 파일이 아직 없습니다. 앱에서 미래 반기 편입/편출 결과를 보여줄 원본이 없는 상태입니다.")
else:
    messages.append("1. 2026_H1 weekly_predictions 파일은 존재합니다.")

if not (AUTO_DIR / "yahoo_price_daily.csv").exists():
    messages.append("2. yahoo_price_daily.csv가 없어서 weekly feature build가 실패할 가능성이 큽니다.")
if not (AUTO_DIR / "naver_stock_meta_weekly.csv").exists():
    messages.append("3. naver_stock_meta_weekly.csv가 없어서 weekly feature build가 실패할 가능성이 큽니다.")

if scored_path.exists():
    scored_df = pd.read_csv(scored_path)
    strong_out_count = int(pd.to_numeric(scored_df.get("strong_out"), errors="coerce").fillna(0).eq(1).sum()) if "strong_out" in scored_df.columns else 0
    if strong_out_count == 0:
        messages.append("4. 예측 파일은 있지만 strong_out == 1 종목이 0개입니다. 이 경우 앱의 편출 패널은 비어 보입니다.")
    else:
        messages.append(f"4. 예측 파일 안에 strong_out 종목이 {strong_out_count}개 있습니다. 이 경우 앱에서도 편출 패널이 보여야 합니다.")

for msg in messages:
    print(msg)


1. 2026_H1 weekly_predictions 파일이 아직 없습니다. 앱에서 미래 반기 편입/편출 결과를 보여줄 원본이 없는 상태입니다.
2. yahoo_price_daily.csv가 없어서 weekly feature build가 실패할 가능성이 큽니다.
3. naver_stock_meta_weekly.csv가 없어서 weekly feature build가 실패할 가능성이 큽니다.


## 7. 주간 갱신을 위해 실행해야 하는 명령 정리

아래는 현재 구조상 2026_H1 주간 갱신에 필요한 기본 실행 순서입니다.
노트북에서는 실행하지 않고 명령만 정리합니다.


In [10]:
commands_df = pd.DataFrame(
    {
        "step": [
            "주간 데이터 수집",
            "주간 예측 실행",
            "생성 파일 확인",
        ],
        "command": [
            "python run_weekly_collection.py",
            "python run_weekly_prediction.py --as-of 2026-04-07",
            "data/incoming/auto/weekly_predictions_2026_H1.csv, weekly_strong_out_2026_H1.csv 확인",
        ],
        "purpose": [
            "Yahoo/Naver/DART 기반 raw 데이터 갱신",
            "활성 기간(2026_H1) 피처 생성 후 strong_in/strong_out 예측 생성",
            "앱이 읽을 결과 파일 존재 여부 확인",
        ],
    }
)
commands_df


,step,command,purpose
0,주간 데이터 수집,python run_weekly_collection.py,Yahoo/Naver/DART 기반 raw 데이터 갱신
1,주간 예측 실행,python run_weekly_prediction.py --as-of 2026-0...,활성 기간(2026_H1) 피처 생성 후 strong_in/strong_out 예측 생성
2,생성 파일 확인,data/incoming/auto/weekly_predictions_2026_H1....,앱이 읽을 결과 파일 존재 여부 확인


## 8. 다음 의사결정 포인트

이 노트북을 실행한 뒤 결과에 따라 다음처럼 나누면 됩니다.

- `weekly_predictions_2026_H1.csv`가 없음:
  주간 수집/예측 파이프라인부터 점검
- 예측 파일은 있는데 `strong_out == 1`이 0개:
  모델 출력 규칙 또는 입력 피처 문제 점검
- 예측 파일에 `strong_out`이 있는데 앱에서 안 보임:
  Streamlit의 기간 선택/분기 로직 점검


## 9. 2026_H1 이후 수집 정책안

아래는 `2026_H1` 이후 미래 반기 예측에서 적용할 운영 정책 초안입니다.

### 핵심 변경점

- `float_rate(유동주식비율)`은 더 이상 계산값 `1 - (관계주식 + 자기주식)`만으로 두지 않고,
  **네이버/WiseReport에 표시되는 실제 유동주식비율 값을 최우선 사용**
- `major_holder_ratio`, `treasury_ratio`도 네이버/WiseReport에서 수집한 값을 우선 사용
- 네이버 수집 실패 시에는 **전주 값 fallback**을 허용하되, fallback 발생 사실을 반드시 기록

### 적용 범위

- 과거 반기(`2025_H2` 이전): 기존 historical 데이터/기존 정의 유지
- 미래 반기(`2026_H1` 이후): 주간 자동 수집 기준 새 정책 적용


In [11]:
policy_df = pd.DataFrame(
    [
        {
            "항목": "float_rate",
            "기존 방식": "관계주식 + 자기주식 기반 간접 계산 또는 기존 값 사용",
            "변경 방식": "네이버/WiseReport 실제 표시값 우선 사용",
            "fallback": "전주 값 허용",
            "비고": "미래 반기 주간 예측의 핵심 기준값",
        },
        {
            "항목": "major_holder_ratio",
            "기존 방식": "DART 또는 간접 수집",
            "변경 방식": "네이버/WiseReport 주요주주 비율 사용",
            "fallback": "전주 값 또는 결측 허용",
            "비고": "필수 실패 원인으로 만들지 않음",
        },
        {
            "항목": "treasury_ratio",
            "기존 방식": "DART 또는 간접 수집",
            "변경 방식": "네이버/WiseReport 자사주 비율 사용",
            "fallback": "전주 값 또는 결측 허용",
            "비고": "필수 실패 원인으로 만들지 않음",
        },
        {
            "항목": "예측 실행 조건",
            "기존 방식": "일부 컬럼 누락 시 파이프라인 실패 가능",
            "변경 방식": "핵심 컬럼(price, shares, float_rate) 중심으로 실행 가능 여부 판단",
            "fallback": "핵심 컬럼만 유지되면 진행",
            "비고": "주간 운영 안정성 우선",
        },
    ]
)
policy_df


,항목,기존 방식,변경 방식,fallback,비고
0,float_rate,관계주식 + 자기주식 기반 간접 계산 또는 기존 값 사용,네이버/WiseReport 실제 표시값 우선 사용,전주 값 허용,미래 반기 주간 예측의 핵심 기준값
1,major_holder_ratio,DART 또는 간접 수집,네이버/WiseReport 주요주주 비율 사용,전주 값 또는 결측 허용,필수 실패 원인으로 만들지 않음
2,treasury_ratio,DART 또는 간접 수집,네이버/WiseReport 자사주 비율 사용,전주 값 또는 결측 허용,필수 실패 원인으로 만들지 않음
3,예측 실행 조건,일부 컬럼 누락 시 파이프라인 실패 가능,"핵심 컬럼(price, shares, float_rate) 중심으로 실행 가능 여부 판단",핵심 컬럼만 유지되면 진행,주간 운영 안정성 우선


## 10. fallback 규칙 초안

`float_rate`를 포함한 네이버 수집값이 비었을 때는 아래 규칙을 추천합니다.


In [12]:
fallback_rule_df = pd.DataFrame(
    [
        {"우선순위": 1, "규칙": "네이버/WiseReport에서 이번 주 실제값 수집 성공", "사용값": "이번 주 실제값", "예측 사용 여부": "사용"},
        {"우선순위": 2, "규칙": "이번 주 수집 실패 + 전주 값 존재", "사용값": "전주 값", "예측 사용 여부": "사용"},
        {"우선순위": 3, "규칙": "이번 주/전주 모두 없음", "사용값": "결측", "예측 사용 여부": "정책 결정 필요"},
    ]
)
fallback_rule_df


,우선순위,규칙,사용값,예측 사용 여부
0,1,네이버/WiseReport에서 이번 주 실제값 수집 성공,이번 주 실제값,사용
1,2,이번 주 수집 실패 + 전주 값 존재,전주 값,사용
2,3,이번 주/전주 모두 없음,결측,정책 결정 필요


## 11. 수집 오류를 확인할 수 있어야 하는 지점

운영 관점에서는 “값을 못 가져왔다”보다 “그 사실을 우리가 볼 수 있느냐”가 더 중요합니다.
아래와 같은 로그/산출물을 남기는 방향이 좋습니다.


In [13]:
monitoring_df = pd.DataFrame(
    [
        {
            "확인 위치": "weekly_collection_summary.json",
            "남길 내용": "총 종목 수, 성공 수, 실패 수, fallback 사용 수",
            "용도": "실행 한 번에 전체 상태 요약",
        },
        {
            "확인 위치": "weekly_collection_errors.csv",
            "남길 내용": "ticker, 항목명, 에러 메시지, fallback 사용 여부",
            "용도": "어느 종목에서 어떤 수집 실패가 났는지 확인",
        },
        {
            "확인 위치": "naver_stock_meta_weekly.csv",
            "남길 내용": "float_rate_source, major_holder_source, treasury_source",
            "용도": "실제값/전주값/기타 fallback 출처 확인",
        },
        {
            "확인 위치": "Streamlit 요약 카드",
            "남길 내용": "이번 주 fallback 사용 종목 수",
            "용도": "앱 화면에서 운영 이상 징후 확인",
        },
    ]
)
monitoring_df


,확인 위치,남길 내용,용도
0,weekly_collection_summary.json,"총 종목 수, 성공 수, 실패 수, fallback 사용 수",실행 한 번에 전체 상태 요약
1,weekly_collection_errors.csv,"ticker, 항목명, 에러 메시지, fallback 사용 여부",어느 종목에서 어떤 수집 실패가 났는지 확인
2,naver_stock_meta_weekly.csv,"float_rate_source, major_holder_source, treasu...",실제값/전주값/기타 fallback 출처 확인
3,Streamlit 요약 카드,이번 주 fallback 사용 종목 수,앱 화면에서 운영 이상 징후 확인


## 12. 지금 기준으로 추가로 확정해야 할 의사결정

아래 항목은 코드 반영 전에 한 번 정해두면 좋습니다.


In [14]:
decision_points_df = pd.DataFrame(
    [
        {"의사결정 항목": "전주 fallback 허용 기간", "현재 추천": "1주", "설명": "오래된 값이 누적 사용되지 않도록 제한"},
        {"의사결정 항목": "float_rate 결측 시 예측 실행 여부", "현재 추천": "전주 값 있으면 실행, 없으면 경고", "설명": "핵심 컬럼이므로 완전 결측은 경고 필요"},
        {"의사결정 항목": "major_holder_ratio / treasury_ratio 결측 시 처리", "현재 추천": "예측은 진행", "설명": "미래 반기 주간 운영에서는 비핵심으로 완화"},
        {"의사결정 항목": "fallback 종목 표시 여부", "현재 추천": "표시", "설명": "운영 신뢰성 확보"},
        {"의사결정 항목": "과거 반기 로직 변경 여부", "현재 추천": "변경 안 함", "설명": "백테스트 일관성 유지"},
    ]
)
decision_points_df


,의사결정 항목,현재 추천,설명
0,전주 fallback 허용 기간,1주,오래된 값이 누적 사용되지 않도록 제한
1,float_rate 결측 시 예측 실행 여부,"전주 값 있으면 실행, 없으면 경고",핵심 컬럼이므로 완전 결측은 경고 필요
2,major_holder_ratio / treasury_ratio 결측 시 처리,예측은 진행,미래 반기 주간 운영에서는 비핵심으로 완화
3,fallback 종목 표시 여부,표시,운영 신뢰성 확보
4,과거 반기 로직 변경 여부,변경 안 함,백테스트 일관성 유지
